# Calpella Early Stopping Experiment

**Objective:** Compare early stopping strategies vs fixed epochs

**Phases:**
1. Quick comparison: baseline (no ES) vs patience mode
2. Mode comparison: patience vs slope variants
3. Analysis: NSE, epochs used, training time

**Settings:**
- Basin: Calpella (daily)
- Max epochs: 48
- Both: no-physics and physics-informed

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import time
from datetime import datetime

# Add project root to path
PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from UCB_training.UCB_train import UCB_trainer
from UCB_training.UCB_utils import save_hparams, load_hparams
from UCB_training.UCB_plotting import plot_loss_curves
from UCB_training.constants import CALPELLA_DAILY_FEATURES, CALPELLA_DAILY_PHYSICS_FEATURES

print(f"Project root: {PROJECT_ROOT}")

ModuleNotFoundError: No module named 'ruamel'

In [ ]:
# Configuration
BASIN = "calpella"
RESOLUTION = "daily"
DATA_DIR = PROJECT_ROOT / "russian_river_data"
YAML_PATH = PROJECT_ROOT / "UCB_training" / "configs" / "calpella_gage_nlayer.yaml"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / BASIN / f"{RESOLUTION}_shared"

# Experiment label
EXPERIMENT_LABEL = "EARLY_STOP_EXP"
RUN_STAMP = datetime.now().strftime("%Y%m%dT%H%M%SZ")

# Create output directories
RUNS_DIR = OUTPUT_DIR / "runs" / f"{EXPERIMENT_LABEL}_{RUN_STAMP}"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {RUNS_DIR}")

In [ ]:
# Base hyperparameters (from baseline best results)
BASE_HYPERPARAMS = {
    'hidden_size': 128,
    'output_dropout': 0.4,
    'seq_length': 120,
    'num_layers': 1,
    'epochs': 48,  # Max epochs - early stopping may stop earlier
    'batch_size': 64,
    'learning_rate': {0: 0.01, 24: 0.005, 36: 0.001},
    'validate_every': 5,
    'save_weights_every': 48,  # Save at end only (early stopping saves on stop)
}

In [ ]:
# Early stopping configurations to test
ES_CONFIGS = {
    # Baseline: No early stopping
    'baseline': {
        'early_stopping': False,
    },
    
    # Patience mode variants
    'patience_p3': {
        'early_stopping': True,
        'early_stopping_mode': 'patience',
        'patience_early_stopping': 3,
        'min_delta_early_stopping': 0.0,
        'minimum_epochs_before_early_stopping': 10,
    },
    'patience_p5': {
        'early_stopping': True,
        'early_stopping_mode': 'patience',
        'patience_early_stopping': 5,
        'min_delta_early_stopping': 0.0,
        'minimum_epochs_before_early_stopping': 10,
    },
    
    # Slope mode variants
    'slope_w7_p2': {
        'early_stopping': True,
        'early_stopping_mode': 'slope',
        'early_stopping_slope_window': 7,
        'early_stopping_slope_patience': 2,
        'early_stopping_slope_min_epoch': 10,
        'early_stopping_slope_ema_alpha': 0.4,
        'early_stopping_slope_eps_slope': 1e-3,
        'minimum_epochs_before_early_stopping': 10,
    },
    'slope_w5_p3': {
        'early_stopping': True,
        'early_stopping_mode': 'slope',
        'early_stopping_slope_window': 5,
        'early_stopping_slope_patience': 3,
        'early_stopping_slope_min_epoch': 10,
        'early_stopping_slope_ema_alpha': 0.4,
        'early_stopping_slope_eps_slope': 1e-3,
        'minimum_epochs_before_early_stopping': 10,
    },
    'slope_w10_p2': {
        'early_stopping': True,
        'early_stopping_mode': 'slope',
        'early_stopping_slope_window': 10,
        'early_stopping_slope_patience': 2,
        'early_stopping_slope_min_epoch': 15,
        'early_stopping_slope_ema_alpha': 0.3,
        'early_stopping_slope_eps_slope': 5e-4,
        'minimum_epochs_before_early_stopping': 15,
    },
}

print(f"Testing {len(ES_CONFIGS)} early stopping configurations")

In [ ]:
def run_experiment(es_config_name, es_config, physics_informed=False, verbose=True):
    """
    Run a single experiment with given early stopping config.
    
    Returns dict with: config_name, physics, epochs_used, training_time, metrics
    """
    model_type = 'physics' if physics_informed else 'no_physics'
    run_name = f"{es_config_name}_{model_type}"
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Running: {run_name}")
        print(f"Early stopping: {es_config}")
        print(f"{'='*60}")
    
    # Merge base hyperparams with ES config
    hyperparams = {**BASE_HYPERPARAMS, **es_config}
    
    # Select features
    features = CALPELLA_DAILY_PHYSICS_FEATURES if physics_informed else CALPELLA_DAILY_FEATURES
    
    # Physics data file
    physics_file = DATA_DIR / "Calpella_daily_shift.csv" if physics_informed else None
    
    # Create trainer
    trainer = UCB_trainer(
        path_to_csv_folder=DATA_DIR,
        yaml_path=YAML_PATH,
        hyperparams=hyperparams,
        input_features=features,
        num_ensemble_members=1,
        physics_informed=physics_informed,
        physics_data_file=physics_file,
        hourly=False,
        gpu=0,
        verbose=verbose,
        runs_parent=RUNS_DIR / run_name,
        run_label=run_name,
        run_stamp=RUN_STAMP,
    )
    
    # Train and time it
    start_time = time.time()
    model_path = trainer.train()
    training_time = time.time() - start_time
    
    # Get actual epochs used
    epochs_used = trainer._get_last_epoch(model_path)
    
    # Get validation metrics
    csv_path, metrics = trainer.results(period='validation')
    
    # Also get test metrics
    trainer._eval_model(model_path, period='test')
    test_csv, test_metrics = trainer.results(period='test')
    
    result = {
        'config_name': es_config_name,
        'physics': physics_informed,
        'model_type': model_type,
        'epochs_configured': BASE_HYPERPARAMS['epochs'],
        'epochs_used': epochs_used,
        'epochs_saved': BASE_HYPERPARAMS['epochs'] - epochs_used,
        'training_time_sec': training_time,
        'model_path': str(model_path),
        'val_NSE': metrics.get('NSE', np.nan),
        'val_KGE': metrics.get('KGE', np.nan),
        'val_RMSE': metrics.get('RMSE', np.nan),
        'test_NSE': test_metrics.get('NSE', np.nan),
        'test_KGE': test_metrics.get('KGE', np.nan),
        'test_RMSE': test_metrics.get('RMSE', np.nan),
        **{f'es_{k}': v for k, v in es_config.items()},
    }
    
    if verbose:
        print(f"\nResults for {run_name}:")
        print(f"  Epochs: {epochs_used}/{BASE_HYPERPARAMS['epochs']} (saved {result['epochs_saved']})")
        print(f"  Training time: {training_time:.1f}s")
        print(f"  Val NSE: {metrics.get('NSE', np.nan):.4f}")
        print(f"  Test NSE: {test_metrics.get('NSE', np.nan):.4f}")
    
    return result, trainer

## Phase 1: Quick Comparison (Baseline vs Patience)

Run baseline and patience_p3 for both physics modes

In [ ]:
# Phase 1: Quick comparison
phase1_configs = ['baseline', 'patience_p3']
phase1_results = []

for config_name in phase1_configs:
    es_config = ES_CONFIGS[config_name]
    
    # No physics
    result, trainer = run_experiment(config_name, es_config, physics_informed=False)
    phase1_results.append(result)
    
    # Physics
    result, trainer = run_experiment(config_name, es_config, physics_informed=True)
    phase1_results.append(result)

phase1_df = pd.DataFrame(phase1_results)
print("\n" + "="*80)
print("PHASE 1 RESULTS")
print("="*80)
display(phase1_df[['config_name', 'model_type', 'epochs_used', 'epochs_saved', 
                   'training_time_sec', 'val_NSE', 'test_NSE']])

## Phase 2: Full Mode Comparison

Run all early stopping configurations

In [ ]:
# Phase 2: All configs (skip baseline and patience_p3 already run)
phase2_configs = ['patience_p5', 'slope_w7_p2', 'slope_w5_p3', 'slope_w10_p2']
phase2_results = []

for config_name in phase2_configs:
    es_config = ES_CONFIGS[config_name]
    
    # No physics
    result, trainer = run_experiment(config_name, es_config, physics_informed=False)
    phase2_results.append(result)
    
    # Physics
    result, trainer = run_experiment(config_name, es_config, physics_informed=True)
    phase2_results.append(result)

phase2_df = pd.DataFrame(phase2_results)
print("\n" + "="*80)
print("PHASE 2 RESULTS")
print("="*80)
display(phase2_df[['config_name', 'model_type', 'epochs_used', 'epochs_saved',
                   'training_time_sec', 'val_NSE', 'test_NSE']])

## Combined Analysis

In [ ]:
# Combine all results
all_results = pd.concat([phase1_df, phase2_df], ignore_index=True)

# Save results
results_path = RUNS_DIR / "early_stopping_experiment_results.csv"
all_results.to_csv(results_path, index=False)
print(f"Results saved to: {results_path}")

# Display full results
display(all_results)

In [ ]:
# Analysis: Compare by model type
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, model_type in enumerate(['no_physics', 'physics']):
    df_sub = all_results[all_results['model_type'] == model_type].copy()
    df_sub = df_sub.sort_values('test_NSE', ascending=False)
    
    # NSE comparison
    ax1 = axes[idx, 0]
    x = range(len(df_sub))
    width = 0.35
    ax1.bar([i - width/2 for i in x], df_sub['val_NSE'], width, label='Validation', alpha=0.8)
    ax1.bar([i + width/2 for i in x], df_sub['test_NSE'], width, label='Test', alpha=0.8)
    ax1.set_xticks(x)
    ax1.set_xticklabels(df_sub['config_name'], rotation=45, ha='right')
    ax1.set_ylabel('NSE')
    ax1.set_title(f'{model_type.upper()}: NSE by Config')
    ax1.legend()
    ax1.axhline(y=df_sub[df_sub['config_name']=='baseline']['test_NSE'].values[0], 
                color='r', linestyle='--', alpha=0.5, label='Baseline')
    
    # Epochs used vs training time
    ax2 = axes[idx, 1]
    colors = ['red' if c == 'baseline' else 'blue' for c in df_sub['config_name']]
    ax2.scatter(df_sub['epochs_used'], df_sub['training_time_sec'], 
                c=colors, s=100, alpha=0.7)
    for i, row in df_sub.iterrows():
        ax2.annotate(row['config_name'], (row['epochs_used'], row['training_time_sec']),
                     fontsize=8, ha='left')
    ax2.set_xlabel('Epochs Used')
    ax2.set_ylabel('Training Time (s)')
    ax2.set_title(f'{model_type.upper()}: Efficiency')

plt.tight_layout()
plt.savefig(RUNS_DIR / "early_stopping_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
print("\n" + "="*80)
print("SUMMARY: Early Stopping Impact")
print("="*80)

for model_type in ['no_physics', 'physics']:
    df_sub = all_results[all_results['model_type'] == model_type]
    baseline = df_sub[df_sub['config_name'] == 'baseline'].iloc[0]
    
    print(f"\n{model_type.upper()}:")
    print(f"  Baseline: NSE={baseline['test_NSE']:.4f}, epochs={baseline['epochs_used']}, time={baseline['training_time_sec']:.1f}s")
    
    for _, row in df_sub[df_sub['config_name'] != 'baseline'].iterrows():
        nse_diff = row['test_NSE'] - baseline['test_NSE']
        time_saved = baseline['training_time_sec'] - row['training_time_sec']
        time_pct = (time_saved / baseline['training_time_sec']) * 100
        
        print(f"  {row['config_name']}: NSE={row['test_NSE']:.4f} ({nse_diff:+.4f}), "
              f"epochs={row['epochs_used']}, time saved={time_saved:.1f}s ({time_pct:.1f}%)")

In [ ]:
# Best config recommendation
print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)

for model_type in ['no_physics', 'physics']:
    df_sub = all_results[all_results['model_type'] == model_type]
    baseline = df_sub[df_sub['config_name'] == 'baseline'].iloc[0]
    
    # Find best: highest NSE with time savings
    es_only = df_sub[df_sub['config_name'] != 'baseline'].copy()
    es_only['nse_delta'] = es_only['test_NSE'] - baseline['test_NSE']
    es_only['time_saved_pct'] = (baseline['training_time_sec'] - es_only['training_time_sec']) / baseline['training_time_sec'] * 100
    
    # Best = minimal NSE loss with good time savings
    es_only['score'] = es_only['nse_delta'] * 100 + es_only['time_saved_pct'] * 0.1  # Weight NSE more
    best = es_only.loc[es_only['score'].idxmax()]
    
    print(f"\n{model_type.upper()}: Best config = {best['config_name']}")
    print(f"  NSE change: {best['nse_delta']:+.4f}")
    print(f"  Time saved: {best['time_saved_pct']:.1f}%")
    print(f"  Epochs: {best['epochs_used']}/{baseline['epochs_used']}")

## Loss Curves Analysis

Visualize where early stopping triggered

In [ ]:
# Plot loss curves for each run
for _, row in all_results.iterrows():
    model_path = Path(row['model_path'])
    if model_path.exists():
        print(f"\nLoss curves for: {row['config_name']} ({row['model_type']})")
        print(f"Epochs used: {row['epochs_used']}")
        try:
            plot_loss_curves(model_path)
        except Exception as e:
            print(f"Could not plot: {e}")